# 01 — Validação da base

**Objetivo:** antes de qualquer análise, conferir se a base reproduz os
números que a ANAC publica. Se não bater, não dá pra avançar: é preciso
saber se a diferença é de escopo, de revisão ou erro meu. Nesta ordem:

1. Os totais: passageiros, ocupação, voos e pontualidade, contra o
   Anuário do Transporte Aéreo e os comunicados mensais da ANAC.

Faz parte da [etapa 4](../README.md#etapas) do projeto. Os dados vêm de
`dados/marts/validacao.parquet`, que o `construir.py` monta a partir da
coleta de 13/10/2026: um registro por mês e empresa, só com somas e
contagens, pra toda média sair ponderada da razão entre as somas.

In [1]:
import pandas as pd

URL = "https://raw.githubusercontent.com/ivadivad/voos-em-numeros/main/dados/marts/validacao.parquet"
mart = pd.read_parquet(URL)
mart["mes"] = pd.to_datetime(mart["mes"])
mart["ano"] = mart["mes"].dt.year
print(f"{len(mart)} linhas, {mart['mes'].nunique()} meses "
      f"({mart['mes'].min():%m/%Y} a {mart['mes'].max():%m/%Y}), {mart['empresa'].nunique()} empresas")

1188 linhas, 92 meses (01/2019 a 08/2026), 24 empresas


## 1. Passageiros, ocupação e pontualidade

O que a ANAC publicou, nas palavras dela:

> "No mercado doméstico de passageiros, foram transportados 101 milhões de
> passageiros em 2025, aumento de 8,4% em relação a 2024. A taxa de ocupação
> das aeronaves atingiu 83,6% no mesmo período." — [notícia do Anuário do
> Transporte Aéreo, 07/07/2026](https://www.gov.br/anac/pt-br/noticias/2026/confira-a-nova-edicao-do-anuario-do-transporte-aereo-da-anac)

> "No mercado doméstico, foram transportados 95,3 milhões de passageiros
> pagos no ano passado, 1,7% a mais que o verificado em 2018." —
> [ANAC Informa de janeiro de 2020](https://www.gov.br/anac/pt-br/noticias/newsletter/2020/anac_informa_janeiro_2020.pdf), sobre 2019

Os passageiros vêm da base estatística, a mesma fonte do Anuário. A
`stg_estatisticas` traz os três grupos de voo (regular, não regular e
improdutivo), e a conta abaixo soma os três.

In [2]:
pax = mart.groupby("ano")["passageiros_pagos"].sum() / 1e6

def compara(medida, base, publicado):
    print(f"{medida:30} base {base:8.2f}   publicado {publicado:7.2f}   "
          f"diferença {base - publicado:+6.2f} ({base / publicado - 1:+.2%})")

compara("passageiros 2019 (milhões)", pax[2019], 95.3)
compara("passageiros 2025 (milhões)", pax[2025], 101)
compara("variação 2025/2024 (%)", (pax[2025] / pax[2024] - 1) * 100, 8.4)

a = mart[mart["ano"] == 2025].sum(numeric_only=True)
compara("ocupação 2025 (%)", a.rpk / a.ask * 100, 83.6)

passageiros 2019 (milhões)     base    95.06   publicado   95.30   diferença  -0.24 (-0.25%)
passageiros 2025 (milhões)     base   101.42   publicado  101.00   diferença  +0.42 (+0.41%)
variação 2025/2024 (%)         base     8.44   publicado    8.40   diferença  +0.04 (+0.42%)
ocupação 2025 (%)              base    83.54   publicado   83.60   diferença  -0.06 (-0.07%)


**2025 bate.** 101,42 milhões contra os "101 milhões" arredondados, e
a variação sobre 2024 sai 8,44%, contra 8,4%. A ocupação, RPK sobre ASK,
fica 0,06 ponto abaixo dos 83,6%.

**2019 fica 0,25% abaixo**, 240 mil passageiros num ano de 95 milhões.
O boletim foi escrito em jan/2020, com os dados de então, e a ANAC
republica a série (o `DICIONARIO.md` registra arquivos substituídos e o
acumulado regerado a cada coleta). O mesmo cálculo bater em 2025, com
número publicado há três meses, aponta pra revisão do dado, e não pra
escopo diferente: escopo erraria nos dois anos.

### Os comunicados mensais

> "9,2 milhões no segmento doméstico (1,9% acima do registrado em julho de
> 2025)" — [ANAC, 20/08/2026](https://www.gov.br/anac/pt-br/noticias/2026/setor-aereo-movimenta-11-9-milhoes-de-passageiros-no-pais-no-mes-de-julho)

> "foram 8,9 milhões de passageiros (crescimento de 2,3%) no segmento
> doméstico" — [ANAC, 21/09/2026](https://www.gov.br/anac/pt-br/noticias/2026/aviacao-civil-brasileira-movimentou-11-5-milhoes-de-passageiros-em-agosto), sobre agosto

In [3]:
mensal = mart.groupby("mes")[["passageiros_pagos", "passageiros_pagos_regular"]].sum() / 1e6
for mes, ano_antes, publicado, variacao in [("2026-07", "2025-07", 9.2, 1.9), ("2026-08", "2025-08", 8.9, 2.3)]:
    atual, antes = mensal.loc[pd.Timestamp(mes)], mensal.loc[pd.Timestamp(ano_antes)]
    compara(f"passageiros {mes} (milhões)", atual.passageiros_pagos, publicado)
    compara(f"variação {mes} (%)", (atual.passageiros_pagos / antes.passageiros_pagos - 1) * 100, variacao)
    compara(f"variação {mes}, só regular (%)",
            (atual.passageiros_pagos_regular / antes.passageiros_pagos_regular - 1) * 100, variacao)

passageiros 2026-07 (milhões)  base     9.21   publicado    9.20   diferença  +0.01 (+0.12%)
variação 2026-07 (%)           base     1.82   publicado    1.90   diferença  -0.08 (-4.27%)
variação 2026-07, só regular (%) base     3.87   publicado    1.90   diferença  +1.97 (+103.77%)
passageiros 2026-08 (milhões)  base     8.94   publicado    8.90   diferença  +0.04 (+0.46%)
variação 2026-08 (%)           base     2.12   publicado    2.30   diferença  -0.18 (-7.71%)
variação 2026-08, só regular (%) base     3.16   publicado    2.30   diferença  +0.86 (+37.28%)


**O volume do mês bate; a variação fica 0,1 a 0,2 ponto abaixo.** Julho
dá 9,21 milhões e agosto 8,94, os dois arredondando pro publicado. Já o
crescimento sobre o mesmo mês de 2025 sai 1,8% e 2,1%, contra 1,9% e
2,3%. Contar só os voos regulares não explica (afasta mais: 3,9% e
3,2%), e o comunicado não traz o número de 2025 pra conferir. Fica como
diferença registrada, pequena demais pra mudar qualquer conclusão, e não
resolvida.

### Pontualidade, do VRA

> "No mercado doméstico, em 2025, 7,1% dos voos registraram atraso superior
> a 30 minutos na chegada, 2,7% tiveram atraso superior a 60 minutos e 1,8%
> das operações foram canceladas." — [notícia do Anuário](https://www.gov.br/anac/pt-br/noticias/2026/confira-a-nova-edicao-do-anuario-do-transporte-aereo-da-anac)

O comunicado não diz o denominador. A conta usa os dois candidatos: os
voos realizados e todos os voos (realizados, cancelados e com situação não
informada).

In [4]:
for minutos, publicado in [(30, 7.1), (60, 2.7)]:
    atrasados = a[f"voos_atraso_{minutos}"]
    compara(f"atraso > {minutos} min / realizados (%)", atrasados / a.voos_realizados * 100, publicado)
    compara(f"atraso > {minutos} min / todos (%)", atrasados / a.voos * 100, publicado)
compara("cancelados / todos (%)", a.voos_cancelados / a.voos * 100, 1.8)

atraso > 30 min / realizados (%) base     7.12   publicado    7.10   diferença  +0.02 (+0.28%)
atraso > 30 min / todos (%)    base     7.00   publicado    7.10   diferença  -0.10 (-1.45%)
atraso > 60 min / realizados (%) base     2.77   publicado    2.70   diferença  +0.07 (+2.71%)
atraso > 60 min / todos (%)    base     2.73   publicado    2.70   diferença  +0.03 (+0.94%)
cancelados / todos (%)         base     1.72   publicado    1.80   diferença  -0.08 (-4.30%)


**Os três batem dentro de 0,1 ponto, mas nenhum denominador reproduz os
dois atrasos ao mesmo tempo.** Sobre os realizados, o atraso de 30 min
dá 7,12% (o 7,1% publicado) e o de 60 min, 2,77%. Sobre todos os voos, o
de 60 min dá 2,73% (o 2,7%) e o de 30 min, 7,00%. Os cancelados ficam em
1,72%, contra 1,8%.

A diferença cabe nos filtros que a staging aplica e que a ANAC pode não
aplicar (os voos com tipo de linha `NA`, todos cancelados, ficam de fora
aqui; ver `DICIONARIO.md`). O VRA, que é a base da pontualidade da etapa
9, reproduz o publicado na casa do décimo de ponto.